**_**CREATING** THE DATA MODEL STAR SCHEMA_**

In [0]:
class Gold_dimesions:
  def __init__(self,spark):
    self.spark = spark
  def new_dimension(self):
    new_query = """
    SELECT
      ROW_NUMBER() OVER(ORDER BY c.coustmer_id) AS coustomer_sk,
      c.coustmer_id,
      c.coustmer_key,
      c.coustmer_firstname,
      c.coustmer_lastname,
      c.marriage_status,
      CASE
          WHEN c.coustmer_gender <> 'n/a' THEN c.coustmer_gender
          ELSE COALESCE(a.gender,'n/a')
      END AS gender,
      c.coustmer_join_date,
      a.birth_date,
      z.CNTRY AS country
    FROM 
      lakehouse.sliver.cleaned_coustmer c
    LEFT JOIN 
      lakehouse.sliver.cust_az a
    ON
      c.coustmer_key = a.customer_number
    LEFT JOIN
      lakehouse.sliver.coustmer_az z
    ON
      c.coustmer_key = z.cid
    """
    #runing the sql query
    print("reading the data")
    new_data = self.spark.sql(new_query)
    #sending the data to the gold
    new_data.write.format("delta").mode("append").saveAsTable("lakehouse.gold.coust_dimension")
    print("sending the data to the gold layer coust_dimension")

In [0]:
gold_dim = Gold_dimesions(spark)
gold_dim.new_dimension()